In [24]:
%pip install -q -U langchain langchain-core langchain-google-genai langchain-community pydantic

# **setup and model connection **

In [25]:
import os
from getpass import getpass

from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

try :
  os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception :
  os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key")

llm=ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)
response = llm.invoke(
    "Explain smart agriculture in one sentence ."
)
print(response.content)

[{'type': 'text', 'text': '**Smart agriculture** is the use of advanced technologies—such as IoT sensors, AI, and data analytics—to optimize farming practices, increase crop yields, and minimize resource waste.', 'extras': {'signature': 'EpYNCpMNAWkUfRPMHsmtBYl4fn3Uc+tgP2BhyOO3EUCFx7zK0b6uwf9I1G3FTWriEqUDL5E6qJSwHKtTR9e2wtKvyDoV0VlzPYOb/GOhQ51AFgeRmyOrnJIp8w4LyQ5wKn7mUsbex2i6NyNXwNUdsE74w76itTQAWJHsFAR8WQ2k4bnyGh4g8Wolw8aw9h4xaqMILP5nbSWQqxQohK+r8Bc164mxqH+JW8n9vQq+68ZHhflKT0g7XT/Hxcay6ZpS23JeO7KXsBamibTKoGXDMHca9r0wx1txGfW7tkk8cTK5HEAj6+cUtCjRUEf72ywGwObl+ZKBt+4Dlyo11EsP9Xio6MdoCP/swrsbeEoHKCbSuhI4JHcO2YR5PFhHtXNH/KcqKPc17fWiDF1Gv0+mPAaxW8hA/kFNn6cZngdJQooYRxJyqARID9/wz5EWZ8MafblzTbHtcT8IPaURZjUjSWoXgTeUrNVAiKqKWjqqqeqisj7nFdxhhcSImBYpymPwlPNqOjiPjf1UGEXFJ0AiOeL7w2YH6pcohZcK1eECb1PoZrKE4P94b+YLgVHb56ckOoXlYOJC2UgO/sc2V2lT/ADolj3qlfPQKFKa69kiEA3benIX84INKjsiWnrtC8RsuE3fsriVBpjh1Bd8UZMwok0rqbOEqBKWUn2tqOUKGLeolYGvvned6oJWizJBP6rqFe6r61hDYGJGGEqZrTCu62Ce7wjkXM+Yn2ugbI8oCMCajccaw+iTYh9kOi

Part A — LCEL Chain

In this section, we build a Smart Agriculture Q&A assistant using LangChain Expression Language (LCEL).

The chain accepts a topic and a response depth ("brief" or "detailed"). A "RunnableLambda" dynamically converts the requested depth into instructions, which are inserted into a custom system prompt.

We test the chain using both "invoke()" for a single request and "batch()" for multiple requests.

In [26]:
from typing import Any

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser


def format_depth(inputs: dict[str, str]) -> dict[str, str]:
    """Validate the inputs and generate instructions for the requested depth."""

    topic = inputs["topic"].strip()
    depth = inputs["depth"].strip().lower()

    if not topic:
        raise ValueError("Topic cannot be empty.")

    if depth not in {"brief", "detailed"}:
        raise ValueError("Depth must be 'brief' or 'detailed'.")

    if depth == "brief":
        instruction = (
            "Give a concise answer in 2-4 sentences. "
            "Focus on the most important information."
        )
    else:
        instruction = (
            "Give a comprehensive explanation with clear sections, "
            "practical examples, benefits, limitations, and useful advice."
        )

    return {
        "topic": topic,
        "depth": depth,
        "depth_instruction": instruction,
    }


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Smart Agriculture Q&A Assistant.

Your role is to provide accurate, practical, and understandable
information about agriculture, irrigation, crops, soil, and farming.

Adapt your answer to these instructions:
{depth_instruction}

If a question requires information that you do not know,
state the uncertainty instead of inventing facts."""
    ),
    (
        "human",
        "Explain the following agricultural topic: {topic}"
    ),
])

chain = (
    RunnableLambda(format_depth)
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL chain created successfully!")

LCEL chain created successfully!


# **Testing invoke()**

In [27]:
brief_result = chain.invoke(
    {
        "topic": " Drip irrigation ",
        "depth": "brief"
    }
)

print(brief_result)

Drip irrigation is a micro-irrigation method that delivers water and nutrients slowly and directly to the plant roots through a network of pipes, tubing, and emitters. By targeting the root zone directly, it minimizes evaporation, runoff, and weed growth, making it one of the most water-efficient irrigation techniques available. This system is especially beneficial in arid regions or for crops that require precise moisture control to maximize yield.


In [28]:
detailed_result = chain.invoke(
    {
        "topic": " Drip irrigation ",
        "depth": "detailed"
    }
)

print(detailed_result)

### What is Drip Irrigation?

**Drip irrigation** (also referred to as trickle or micro-irrigation) is a low-pressure, high-efficiency water delivery system. It works by delivering water slowly and directly to the plant’s root zone through a network of pipes, tubing, valves, and emitters (drippers). 

Unlike conventional flood or overhead sprinkler methods that wet the entire field surface and foliage, drip irrigation applies water only where the plant needs it most. This minimizes evaporation, surface runoff, and deep percolation below the root zone.

---

### How the System Works: Key Components

A standard drip irrigation setup consists of several core components working in sequence:

1. **Water Source and Pump:** Supplies the required water volume at a steady, low-to-medium operating pressure (usually 10 to 30 PSI / 0.7 to 2.0 bar).
2. **Filtration System:** The "heart" of the system. It prevents suspended particles, algae, and minerals from clogging the small emitter openings. Com

# **Testing Batch()**

In [29]:
questions: list[dict[str, str]] = [
    {
        "topic": "Crop rotation",
        "depth": "brief",
    },
    {
        "topic": "Soil salinity management",
        "depth": "detailed",
    },
    {
        "topic": "Composting",
        "depth": "brief",
    },
]

batch_results = chain.batch(questions)

for index, result in enumerate(batch_results, start=1):
    print(f"\n--- Answer {index} ---")
    print(result)


--- Answer 1 ---
Crop rotation is the agricultural practice of growing different types of crops sequentially on the same plot of land across different seasons. By alternating crops with varying nutrient requirements and root depths, it naturally improves soil structure and replenishes essential nutrients, such as nitrogen. This practice also disrupts the life cycles of pests, diseases, and weeds, significantly reducing the need for chemical fertilizers and pesticides.

--- Answer 2 ---
---

### What is Soil Salinity?

**Soil salinity** refers to the excessive accumulation of water-soluble salts in the soil profile. The primary salts involved include chlorides, sulfates, carbonates, and bicarbonates of sodium, calcium, and magnesium. 

Soil is generally classified as saline when the **Electrical Conductivity (EC)** of a saturated soil extract exceeds **4 dS/m (deciSiemens per meter)** at 25°C. 

Salinity harms plants in two primary ways:
1. **Osmotic Stress ("Physiological Drought"):**

In [30]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Smart Agriculture Q&A Assistant.

Provide accurate and practical agricultural information.
Use the conversation history to understand follow-up questions.

Adapt your answer to these instructions:
{depth_instruction}

If you are uncertain, explain the uncertainty.
"""
    ),
    MessagesPlaceholder(variable_name="history", optional=True),
    (
        "human",
        "Explain or answer this agricultural question: {topic}"
    ),
])

chain = (
    RunnableLambda(format_depth)
    | prompt
    | llm
    | StrOutputParser()
)

print("Chain updated to support conversation history.")

Chain updated to support conversation history.


In [31]:
import os
import re

from langchain_community.chat_message_histories import (
    FileChatMessageHistory,
)
from langchain_core.runnables.history import (
    RunnableWithMessageHistory,
)


HISTORY_DIR = "04_Feedback/chat_histories"
os.makedirs(HISTORY_DIR, exist_ok=True)


def get_session_history(session_id: str) -> FileChatMessageHistory:
    """Return persistent message history for a session."""

    if not re.fullmatch(r"[A-Za-z0-9_-]+", session_id):
        raise ValueError("Invalid session ID.")

    file_path = os.path.join(
        HISTORY_DIR,
        f"{session_id}.json",
    )

    return FileChatMessageHistory(file_path=file_path)


conversation_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="topic",
    history_messages_key="history",
)

print("Persistent conversation memory configured.")

Persistent conversation memory configured.


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [32]:
session_id = "agriculture_demo_01"

turns: list[dict[str, str]] = [
    {
        "topic": "What is drip irrigation?",
        "depth": "brief",
    },
    {
        "topic": "What are its main benefits for farmers?",
        "depth": "brief",
    },
    {
        "topic": "How can those benefits help conserve water?",
        "depth": "detailed",
    },
    {
        "topic": "Which crops can benefit from this irrigation method?",
        "depth": "brief",
    },
    {
        "topic": "What problems might reduce its effectiveness?",
        "depth": "detailed",
    },
    {
        "topic": (
            "Based on our discussion, summarize the benefits, "
            "limitations, and practical recommendations."
        ),
        "depth": "detailed",
    },
]

config = {
    "configurable": {
        "session_id": session_id,
    }
}

for index, turn in enumerate(turns, start=1):
    print(f"\n{'=' * 20} TURN {index} {'=' * 20}")
    print("User:", turn["topic"])

    answer = conversation_chain.invoke(
        turn,
        config=config,
    )

    print("Assistant:", answer)


==================== TURN 1 ====================
User: What is drip irrigation?
Assistant: Drip irrigation is a micro-irrigation system that delivers water and nutrients slowly and directly to the plant's root zone through a network of tubes, pipes, and emitters. By applying water precisely where it is needed, it minimizes evaporation, deep percolation, and runoff. This highly efficient method conserves water—often saving 30% to 50% compared to conventional methods—while reducing weed growth and optimizing crop yields.

==================== TURN 2 ====================
User: What are its main benefits for farmers?
Assistant: It is unclear which specific agricultural tool, technology, or farming practice you are referring to, as the subject was not specified. If you are asking about smart or precision agriculture in general, the main benefits are higher crop yields, lower input costs (such as water, fertilizer, and labor), and better risk management through data-driven decisions. Please

In [33]:
history = get_session_history(session_id)

print("Total messages:", len(history.messages))
print("\nFULL CONVERSATION HISTORY\n")

for message in history.messages:
    role = (
        "USER"
        if message.type == "human"
        else "ASSISTANT"
        if message.type == "ai"
        else message.type.upper()
    )

    print(f"[{role}]")
    print(message.content)
    print("-" * 60)

print("\nHistory file:", history)

Total messages: 20

FULL CONVERSATION HISTORY

[USER]
What is drip irrigation?
------------------------------------------------------------
[ASSISTANT]
Drip irrigation is a micro-irrigation system that delivers water and nutrients directly to the soil at the plant's root zone. It operates using a network of pipes, tubing, and emitters that release water slowly and steadily under low pressure. By applying water only where needed, this method maximizes water-use efficiency by significantly reducing evaporation, runoff, and weed growth.
------------------------------------------------------------
[USER]
What are its main benefits for farmers?
------------------------------------------------------------
[ASSISTANT]
It is unclear what technology or practice "it" refers to, but assuming you mean smart agriculture, its primary benefits are reduced input costs and maximized crop yields. By utilizing tools like IoT sensors, GPS guidance, and automated irrigation, farmers can apply precise amoun

In [36]:
reloaded_history = get_session_history(session_id)

# Since multiple runs can append messages, we verify we have at least the 12 messages from our session turns
assert len(reloaded_history.messages) >= 12

print("Persistence test passed!")
print("Messages recovered:", len(reloaded_history.messages))

Persistence test passed!
Messages recovered: 20


Part C — Structured Output

In this section, we define a Pydantic model called "Answer" to enforce a consistent response structure.

The model contains a summary, a list of key points, and a confidence score between 0 and 1.

We use "llm.with_structured_output(Answer)" to instruct the language model to return data matching this schema.

In [37]:
from typing import List

from pydantic import BaseModel, Field


class Answer(BaseModel):
    """Structured response for the Smart Agriculture assistant."""

    summary: str = Field(
        description="A concise summary of the answer."
    )

    key_points: List[str] = Field(
        description="A list of the most important points."
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score between 0 and 1."
    )


print(Answer.model_json_schema())

{'description': 'Structured response for the Smart Agriculture assistant.', 'properties': {'summary': {'description': 'A concise summary of the answer.', 'title': 'Summary', 'type': 'string'}, 'key_points': {'description': 'A list of the most important points.', 'items': {'type': 'string'}, 'title': 'Key Points', 'type': 'array'}, 'confidence': {'description': 'Confidence score between 0 and 1.', 'maximum': 1.0, 'minimum': 0.0, 'title': 'Confidence', 'type': 'number'}}, 'required': ['summary', 'key_points', 'confidence'], 'title': 'Answer', 'type': 'object'}


In [38]:
valid_answer = Answer(
    summary="Drip irrigation delivers water near plant roots.",
    key_points=[
        "Can reduce water waste",
        "Delivers water close to the roots",
        "Requires proper maintenance",
    ],
    confidence=0.95,
)

print(valid_answer)
print("\nAs dictionary:")
print(valid_answer.model_dump())

summary='Drip irrigation delivers water near plant roots.' key_points=['Can reduce water waste', 'Delivers water close to the roots', 'Requires proper maintenance'] confidence=0.95

As dictionary:
{'summary': 'Drip irrigation delivers water near plant roots.', 'key_points': ['Can reduce water waste', 'Delivers water close to the roots', 'Requires proper maintenance'], 'confidence': 0.95}


In [39]:
from pydantic import ValidationError

try:
    invalid_answer = Answer(
        summary="Example answer",
        key_points=["Example point"],
        confidence=1.5,
    )
except ValidationError as error:
    print("Validation worked!")
    print(error)

Validation worked!
1 validation error for Answer
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [40]:
structured_llm = llm.with_structured_output(Answer)

structured_response = structured_llm.invoke(
    "Explain the benefits and limitations of drip irrigation."
)

print(structured_response)
print(type(structured_response))

assert isinstance(structured_response, Answer)
assert 0.0 <= structured_response.confidence <= 1.0

summary='Drip irrigation is a highly efficient water delivery method that applies water directly to plant root zones, maximizing water conservation and crop health while presenting challenges related to high upfront costs and maintenance.' key_points=['Benefit: High water efficiency (90%+) by minimizing runoff, deep percolation, and evaporation.', 'Benefit: Precise application of water and fertilizers (fertigation) directly to crop root zones.', 'Benefit: Suppresses weed growth and reduces foliar fungal diseases by keeping inter-row soil and plant foliage dry.', 'Limitation: High initial capital costs for installation, pumps, tubing, and specialized filtration systems.', 'Limitation: Frequent risk of emitter clogging from particulate matter, algae, or mineral precipitates.', 'Limitation: Susceptibility to physical damage from UV radiation, farm machinery, and rodents.'] confidence=0.95
<class '__main__.Answer'>


Part D — Custom Agent

In this section, we build a Smart Agriculture Agent with three tools: a local farming guide search tool, a live weather lookup tool, and an irrigation calculator.

The agent can select and combine tools to answer multi-step questions. We also inspect raw tool calls to understand how the language model requests tool execution.

In [46]:
from pathlib import Path

guide_path = Path("farming_guide.txt")

farming_guide = """
SMART AGRICULTURE FARMING GUIDE

1. Drip Irrigation:
Drip irrigation delivers water directly near plant roots.
It can reduce water waste and help maintain soil moisture.
It is suitable for many crops, including vegetables and fruit trees.
Filters and regular maintenance are important because emitters can clog.

2. Soil Moisture:
Farmers can use soil moisture sensors to decide when irrigation is needed.
Irrigation scheduling should consider crop type, soil, weather, and growth stage.
Over-irrigation can waste water and may harm plant roots.

3. Weather and Irrigation:
Rainfall forecasts can help farmers avoid unnecessary irrigation.
High temperatures can increase crop water demand.
Weather data should be considered alongside soil moisture measurements.

4. Irrigation Efficiency:
Irrigation efficiency describes the fraction of applied water that reaches the
intended crop root zone or is effectively used, depending on the definition.
Higher efficiency can reduce the amount of water that must be applied.
Actual irrigation requirements depend on crop needs, soil, and local conditions.

5. Practical Recommendations:
Inspect irrigation pipes and filters regularly.
Check soil moisture before watering.
Consider recent rainfall and weather forecasts.
Use crop-specific recommendations when deciding irrigation amounts.
"""

guide_path.write_text(farming_guide.strip(), encoding="utf-8")

print("Farming guide created:", guide_path.resolve())
print(guide_path.read_text(encoding="utf-8"))

Farming guide created: /content/farming_guide.txt
SMART AGRICULTURE FARMING GUIDE

1. Drip Irrigation:
Drip irrigation delivers water directly near plant roots.
It can reduce water waste and help maintain soil moisture.
It is suitable for many crops, including vegetables and fruit trees.
Filters and regular maintenance are important because emitters can clog.

2. Soil Moisture:
Farmers can use soil moisture sensors to decide when irrigation is needed.
Irrigation scheduling should consider crop type, soil, weather, and growth stage.
Over-irrigation can waste water and may harm plant roots.

3. Weather and Irrigation:
Rainfall forecasts can help farmers avoid unnecessary irrigation.
High temperatures can increase crop water demand.
Weather data should be considered alongside soil moisture measurements.

4. Irrigation Efficiency:
Irrigation efficiency describes the fraction of applied water that reaches the
intended crop root zone or is effectively used, depending on the definition.
Highe

In [47]:
import re
import requests

from langchain_core.tools import tool


@tool
def search_farming_guide(query: str) -> str:
    """Search the local farming guide for useful agricultural information."""

    text = guide_path.read_text(encoding="utf-8")
    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n\n")
        if paragraph.strip()
    ]

    query_words = set(re.findall(r"\w+", query.lower()))
    scored_paragraphs: list[tuple[int, str]] = []

    for paragraph in paragraphs:
        paragraph_words = set(re.findall(r"\w+", paragraph.lower()))
        score = len(query_words & paragraph_words)

        if score > 0:
            scored_paragraphs.append((score, paragraph))

    scored_paragraphs.sort(key=lambda item: item[0], reverse=True)

    if not scored_paragraphs:
        return "No matching information was found in the local farming guide."

    return "\n\n".join(
        paragraph for _, paragraph in scored_paragraphs[:3]
    )


@tool
def get_weather(city: str) -> str:
    """Get current weather conditions for a city using the Open-Meteo API."""

    try:
        geocoding_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json",
            },
            timeout=15,
        )
        geocoding_response.raise_for_status()

        locations = geocoding_response.json().get("results", [])

        if not locations:
            return f"No location was found for: {city}"

        location = locations[0]
        latitude = location["latitude"]
        longitude = location["longitude"]

        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": (
                    "temperature_2m,relative_humidity_2m,"
                    "precipitation,wind_speed_10m"
                ),
                "timezone": "auto",
            },
            timeout=15,
        )
        weather_response.raise_for_status()

        current = weather_response.json().get("current", {})

        if not current:
            return "Weather data is currently unavailable."

        return (
            f"Location: {location['name']}\n"
            f"Temperature: {current.get('temperature_2m')} °C\n"
            f"Relative humidity: "
            f"{current.get('relative_humidity_2m')}%\n"
            f"Precipitation: {current.get('precipitation')} mm\n"
            f"Wind speed: {current.get('wind_speed_10m')} km/h"
        )

    except (requests.RequestException, ValueError, KeyError) as error:
        return f"Weather API request failed: {error}"


@tool
def calculate_irrigation(
    area_m2: float,
    water_mm: float,
    efficiency: float,
) -> str:
    """Estimate irrigation water volume.

    Args:
        area_m2: Field area in square meters.
        water_mm: Target net water depth in millimeters.
        efficiency: Irrigation efficiency as a fraction from above 0 to 1.
    """

    if area_m2 <= 0:
        return "Error: area_m2 must be greater than zero."

    if water_mm < 0:
        return "Error: water_mm cannot be negative."

    if not 0 < efficiency <= 1:
        return "Error: efficiency must be greater than 0 and at most 1."

    net_liters = area_m2 * water_mm
    gross_liters = net_liters / efficiency

    return (
        f"Field area: {area_m2:.1f} m²\n"
        f"Target net water volume: {net_liters:.1f} liters\n"
        f"Estimated applied water at {efficiency:.0%} efficiency: "
        f"{gross_liters:.1f} liters\n"
        "This is a mathematical estimate, not a crop-specific "
        "irrigation recommendation."
    )


tools = [
    search_farming_guide,
    get_weather,
    calculate_irrigation,
]

print("Available tools:")
for item in tools:
    print(f"- {item.name}: {item.description}")

Available tools:
- search_farming_guide: Search the local farming guide for useful agricultural information.
- get_weather: Get current weather conditions for a city using the Open-Meteo API.
- calculate_irrigation: Estimate irrigation water volume.

Args:
    area_m2: Field area in square meters.
    water_mm: Target net water depth in millimeters.
    efficiency: Irrigation efficiency as a fraction from above 0 to 1.


In [48]:
print("TEST 1: Farming guide")
print(search_farming_guide.invoke({"query": "drip irrigation water"}))

print("\nTEST 2: Weather API")
print(get_weather.invoke({"city": "Banha"}))

print("\nTEST 3: Irrigation calculator")
print(
    calculate_irrigation.invoke({
        "area_m2": 1000,
        "water_mm": 5,
        "efficiency": 0.8,
    })
)

TEST 1: Farming guide
1. Drip Irrigation:
Drip irrigation delivers water directly near plant roots.
It can reduce water waste and help maintain soil moisture.
It is suitable for many crops, including vegetables and fruit trees.
Filters and regular maintenance are important because emitters can clog.

2. Soil Moisture:
Farmers can use soil moisture sensors to decide when irrigation is needed.
Irrigation scheduling should consider crop type, soil, weather, and growth stage.
Over-irrigation can waste water and may harm plant roots.

3. Weather and Irrigation:
Rainfall forecasts can help farmers avoid unnecessary irrigation.
High temperatures can increase crop water demand.
Weather data should be considered alongside soil moisture measurements.

TEST 2: Weather API
Location: Banhā
Temperature: 28.9 °C
Relative humidity: 40%
Precipitation: 0.0 mm
Wind speed: 10.0 km/h

TEST 3: Irrigation calculator
Field area: 1000.0 m²
Target net water volume: 5000.0 liters
Estimated applied water at 80% e

In [49]:
from langchain.agents import create_agent

agriculture_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a Smart Agriculture Assistant.

Your job is to provide practical agricultural guidance.

Rules:
1. Use the farming guide tool when local agricultural information is relevant.
2. Use the weather tool when the user asks about current weather conditions.
3. Use the irrigation calculator for numerical irrigation calculations.
4. You may use multiple tools to answer a single question.
5. Explain the results clearly and mention relevant limitations.
6. Never invent weather measurements or claim a tool succeeded if it failed.
7. A calculated water volume is an estimate, not a crop-specific prescription.
8. If a tool fails, explain the limitation honestly.
""",
)

print("Smart Agriculture Agent created successfully.")

Smart Agriculture Agent created successfully.


In [50]:
query_1 = """
I have a farm near Banha, Egypt.
Check the current weather, search the farming guide for relevant
irrigation advice, and explain what I should consider before irrigating.
"""

result_1 = agriculture_agent.invoke({
    "messages": [
        {"role": "user", "content": query_1}
    ]
})

print("QUESTION 1:")
print(query_1)

print("\nAGENT ANSWER:")
print(result_1["messages"][-1].content)

QUESTION 1:

I have a farm near Banha, Egypt.
Check the current weather, search the farming guide for relevant
irrigation advice, and explain what I should consider before irrigating.


AGENT ANSWER:
[{'type': 'text', 'text': 'Here is the current weather update for Banha, key advice from the farming guide, and what you should consider before irrigating.\n\n---\n\n### 1. Current Weather in Banha, Egypt\n* **Temperature:** 28.5 °C\n* **Relative Humidity:** 41%\n* **Precipitation:** 0.0 mm\n* **Wind Speed:** 9.8 km/h\n\n---\n\n### 2. Relevant Guidance from the Farming Guide\n* **Irrigation Method:** Drip irrigation is highly recommended for vegetables and orchards because it targets the root zone directly and minimizes water loss. Regular maintenance and filtration are required to prevent emitter clogging.\n* **Soil Moisture Monitoring:** Over-irrigation can lead to root rot and wasted water. Scheduling should be based on real soil moisture rather than fixed calendar days alone.\n* **Weat

In [51]:
query_2 = """
I have a 1000 square meter field.
Calculate the irrigation water volume for a target net water depth
of 5 mm and an irrigation efficiency of 80%.
Then consult the farming guide and explain practical precautions.
"""

result_2 = agriculture_agent.invoke({
    "messages": [
        {"role": "user", "content": query_2}
    ]
})

print("QUESTION 2:")
print(query_2)

print("\nAGENT ANSWER:")
print(result_2["messages"][-1].content)

QUESTION 2:

I have a 1000 square meter field.
Calculate the irrigation water volume for a target net water depth
of 5 mm and an irrigation efficiency of 80%.
Then consult the farming guide and explain practical precautions.


AGENT ANSWER:
[{'type': 'text', 'text': '### 1. Irrigation Water Volume Calculation\n\n* **Field Area:** 1,000 m²\n* **Target Net Depth:** 5 mm (equivalent to 5 liters per m²)\n* **Irrigation Efficiency:** 80% (0.80)\n\n**Results:**\n* **Net Water Needed:** 5,000 liters (5.0 m³)\n* **Estimated Total Water to Apply:** **6,250 liters** (6.25 m³)\n\n---\n\n### 2. Practical Precautions from the Farming Guide\n\nWhen planning and carrying out irrigation, consider the following best practices and precautions:\n\n1. **Monitor Soil Moisture to Prevent Over-Irrigation:**\n   * Do not irrigate on a rigid schedule alone. Use soil moisture sensors or manual soil inspection to determine actual water needs.\n   * **Over-irrigation** wastes water and can lead to waterlogging, p

In [53]:
def print_agent_trace(result: dict) -> None:
    """Print agent messages and any tool calls or tool results."""

    for index, message in enumerate(result["messages"]):
        print(f"\n--- Message {index + 1} ---")
        print("Type:", type(message).__name__)

        tool_calls = getattr(message, "tool_calls", None)

        if tool_calls:
            print("Tool calls:")
            for tool_call in tool_calls:
                print(tool_call)

        content = getattr(message, "content", "")
        if content:
            print("Content:", content)

        if getattr(message, "type", "") == "tool":
            print("Tool result:", content)


print("TRACE FOR QUESTION 1")
print_agent_trace(result_1)

print("\nTRACE FOR QUESTION 2")
print_agent_trace(result_2)

TRACE FOR QUESTION 1

--- Message 1 ---
Type: HumanMessage
Content: 
I have a farm near Banha, Egypt.
Check the current weather, search the farming guide for relevant
irrigation advice, and explain what I should consider before irrigating.


--- Message 2 ---
Type: AIMessage
Tool calls:
{'name': 'get_weather', 'args': {'city': 'Banha'}, 'id': 'call_1515015', 'type': 'tool_call'}
{'name': 'search_farming_guide', 'args': {'query': 'irrigation advice'}, 'id': 'call_1515016', 'type': 'tool_call'}

--- Message 3 ---
Type: ToolMessage
Content: Location: Banhā
Temperature: 28.5 °C
Relative humidity: 41%
Precipitation: 0.0 mm
Wind speed: 9.8 km/h
Tool result: Location: Banhā
Temperature: 28.5 °C
Relative humidity: 41%
Precipitation: 0.0 mm
Wind speed: 9.8 km/h

--- Message 4 ---
Type: ToolMessage
Content: 1. Drip Irrigation:
Drip irrigation delivers water directly near plant roots.
It can reduce water waste and help maintain soil moisture.
It is suitable for many crops, including vegetables an

In [54]:
tool_calling_llm = llm.bind_tools(tools)

raw_response = tool_calling_llm.invoke(
    """
    Search the farming guide for drip irrigation advice and calculate
    the water volume for a 1000 m² field, 5 mm net water depth,
    and 80% irrigation efficiency.
    """
)

print("RAW TOOL CALLS:")
print(raw_response.tool_calls)

assert raw_response.tool_calls, (
    "No tool calls were returned. Check model tool-calling support "
    "or try the request again."
)

RAW TOOL CALLS:
[{'name': 'search_farming_guide', 'args': {'query': 'drip irrigation advice'}, 'id': 'call_2841333', 'type': 'tool_call'}, {'name': 'calculate_irrigation', 'args': {'efficiency': 0.8, 'water_mm': 5, 'area_m2': 1000}, 'id': 'call_2841334', 'type': 'tool_call'}]
